In [518]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [519]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

In [520]:
df = pd.read_csv('data/games_list.csv')
df.head(1)

,name,slug,appid,release_date,total_reviews,followers,review_score,avg_playtime,copies_sold,revenue,...,owners,peak_concurrent_players_after_90,peak_concurrent_players_timestamp,avg_concurrent_players_after_90,estimated_launch_reviews,estimated_launch_followers,estimated_launch_copies_sold,developer,publisher,avg_sentiment
0,Half-Life 2,half_life_2,220,2007-10-10,188867.0,100203.0,98.0,11.983311,12989871.0,49366395.0,...,24298922.0,2863.0,2008-10-19 00:00:00,1630.25641,18886.7,10020.3,1298987.1,Valve,Valve,0.453232


In [521]:

df['log_estimated_launch_copies'] = np.log1p(df['estimated_launch_copies_sold'])
df['log_estimated_launch_reviews'] = np.log1p(df['estimated_launch_reviews'])
df['log_estimated_launch_followers'] = np.log1p(df['estimated_launch_followers'])
df['log_avg_playtime'] = np.log1p(df['avg_playtime'])
df['log_review_score'] = np.log1p(df['review_score'])
df['log_avg_concurrent_players_after_90'] = np.log1p(df['avg_concurrent_players_after_90'])


# Log calculations 
df['log_review_to_follower_ratio'] = df['log_estimated_launch_reviews'] - df['log_estimated_launch_followers']
df['log_copies_to_follower_ratio'] = df['log_estimated_launch_copies'] - df['log_estimated_launch_followers']
# ratios for logs are subtraction due to the rules of logs log(a/b) = log(a) - log(b)
df['log_engagement_quality'] = df['log_avg_playtime'] + df['log_review_score']
# log (a*b) = log(a) + log(b)
df['log_virality_factor'] = df['log_review_to_follower_ratio'] + df['log_review_score']

# features we want to train on
features = [
    'log_estimated_launch_copies',
    'log_review_to_follower_ratio',
    'log_copies_to_follower_ratio',
    'log_engagement_quality',
    'log_virality_factor',
    'log_review_score',
    'avg_sentiment'
]





In [522]:
X = df[features]
y = df['log_avg_concurrent_players_after_90']
X.head(1)

,log_estimated_launch_copies,log_review_to_follower_ratio,log_copies_to_follower_ratio,log_engagement_quality,log_virality_factor,log_review_score,avg_sentiment
0,14.077096,0.633798,4.864628,7.158785,5.228918,4.59512,0.453232


In [523]:
X_train, X_test, y_train,y_test = train_test_split(X,y,test_size=TEST_SIZE,random_state=RANDOM_STATE)

In [524]:
# standard random forest without any settings changed
random_forest_base = RandomForestRegressor(random_state=RANDOM_STATE)
random_forest_base.fit(X_train,y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [525]:
y_pred_log = random_forest_base.predict(X_test)

In [526]:
mean_absolute_error(y_test, y_pred_log)

0.8122879125424624

In [527]:
y_test_orig = np.expm1(y_test)
y_pred_orig = np.expm1(y_pred_log)
mean_absolute_error(y_test_orig,y_pred_orig)

22521.531950963574

In [528]:
mean_squared_error(y_test_orig,y_pred_orig)

1555108245.8255293

In [529]:
r2_score(y_test_orig,y_pred_orig)

-0.6671819966029433

In [530]:
params= {
    'n_estimators': [200, 500],
    'max_depth': [10, 15, 20],
    'min_samples_leaf': [1, 2, 5]
}

In [531]:
from sklearn.model_selection import GridSearchCV

In [532]:
random_forest_cv = GridSearchCV(estimator=random_forest_base,param_grid=params,cv=3,scoring='r2',n_jobs=-1)

In [533]:
sample_weights = 1 / np.sqrt(np.expm1(y))  
X_train, X_test, y_train, y_test, sw_train, sw_test = train_test_split(
    X, y, sample_weights,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)
random_forest_cv.fit(X_train,y_train,sample_weight=sw_train)

,estimator,RandomForestR...ndom_state=42)
,param_grid,"{'max_depth': [10, 15, ...], 'min_samples_leaf': [1, 2, ...], 'n_estimators': [200, 500]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [534]:
y_pred_log = random_forest_cv.predict(X_test)

In [535]:
mean_absolute_error(y_test, y_pred_log)

0.7537840192383988

In [536]:
y_test_orig = np.expm1(y_test)
y_pred_orig = np.expm1(y_pred_log)
mean_absolute_error(y_test_orig,y_pred_orig)

14096.198982227266

In [537]:
r2_score(y_test_orig,y_pred_orig)

0.4206115161744508

In [538]:
mean_squared_error(y_pred=y_pred_orig,y_true=y_test_orig)

540439982.299094

In [539]:
X.head(1)

,log_estimated_launch_copies,log_review_to_follower_ratio,log_copies_to_follower_ratio,log_engagement_quality,log_virality_factor,log_review_score,avg_sentiment
0,14.077096,0.633798,4.864628,7.158785,5.228918,4.59512,0.453232


In [540]:
# This piece of code is a proof of concept for trying to predict other games that weren't in the dataset at all

# Data pulled from steamdb and gamalytic (manually) with some guestimates
new_game = {
    'name': 'Dragon ball',
    'estimated_launch_reviews': 6550,
    'estimated_launch_followers': 18284,
    'estimated_launch_copies_sold': 983127,
    'avg_playtime': 12.2,
    'review_score': 63,
    'avg_sentiment': 0.53
}

# Convert to DataFrame
df = pd.DataFrame([new_game])

# Log-transform base features
df['log_estimated_launch_reviews'] = np.log1p(df['estimated_launch_reviews'])
df['log_estimated_launch_followers'] = np.log1p(df['estimated_launch_followers'])
df['log_estimated_launch_copies'] = np.log1p(df['estimated_launch_copies_sold'])
df['log_avg_playtime'] = np.log1p(df['avg_playtime'])
df['log_review_score'] = np.log1p(df['review_score'])

# Engineer features (same as training)
df['log_review_to_follower_ratio'] = df['log_estimated_launch_reviews'] - df['log_estimated_launch_followers']
df['log_copies_to_follower_ratio'] = df['log_estimated_launch_copies'] - df['log_estimated_launch_followers']
df['log_engagement_quality'] = df['log_avg_playtime'] + df['log_review_score']
df['log_virality_factor'] = df['log_review_to_follower_ratio'] + df['log_review_score']

# Select features (same order as training)
features = [
    'log_estimated_launch_copies',
    'log_review_to_follower_ratio',
    'log_copies_to_follower_ratio',
    'log_engagement_quality',
    'log_virality_factor',
    'log_review_score',
    'avg_sentiment'
]

X_new = df[features]

# Predict
y_pred_log = random_forest_cv.predict(X_new)
y_pred_original = np.expm1(y_pred_log)

print(f"\n{'='*60}")
print(f"Prediction for: {new_game['name']}")
print(f"{'='*60}")
print(f"Expected avg concurrent players (90 days): {y_pred_original[0]:,.0f}")
print(f"{'='*60}")



Prediction for: Dragon ball
Expected avg concurrent players (90 days): 2,113
